# ਪਾਠ 03 - ਏਜੰਟਿਕ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ

ਇਸ ਪਾਠ ਵਿੱਚ, ਅਸੀਂ ਪ੍ਰਭਾਵਸ਼ালী AI ਏਜੰਟ ਬਣਾਉਣ ਲਈ ਤਿੰਨ ਬੁਨਿਆਦੀ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨਾਂ ਦੀ ਜਾਂਚ ਕਰਦੇ ਹਾਂ:

1. **ਸਾਫ਼ ਏਜੰਟ ਹਦਾਇਤਾਂ** — ਏਜੇੰਟ ਦੇ ਵਰਤਾਰਾ ਨੂੰ ਦਿੱਖਾਉਣ ਵਾਲੇ ਸਹੀ, ਭੂਮਿਕਾ-ਨਿਰਧਾਰਕ ਪ੍ਰਾਂਪਟ ਬਣਾਉਣਾ
2. **ਪਾਇਡੈਂਟਿਕ ਮਾਡਲ ਨਾਲ ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ** — ਇਹ ਯਕੀਨੀ ਬਣਾਉਣਾ ਕਿ ਏਜੰਟ ਪੂਰਵ-ਦਾੰਸਿਆ ਗਯਾ, ਮੰਨਿਆ ਹੋਇਆ ਡੇਟਾ ਵਾਪਸ ਕਰਦਾ ਹੈ
3. **ਏਕ ਜ਼ਿੰਮੇਵਾਰੀ ਵਾਲੇ ਏਜੰਟ** — ਕੇਂਦਰਿਤ ਏਜੰਟਾਂ ਦੀ ਡਿਜ਼ਾਈਨ ਜਿਹੜੇ ਹਰ ਇੱਕ ਚੀਜ਼ ਚੰਗੀ ਤਰ੍ਹਾਂ ਕਰਦਾ ਹੈ

ਅਸੀਂ ਹਰ ਪੈਟਰਨ ਨੂੰ ਇੱਕ **ਯਾਤਰਾ ਮੰਜ਼ਿਲ ਸਿਫਾਰਸ਼ਕਾਰ** ਸੰਦਰਭ ਵਿੱਚ ਲਾਗੂ ਕਰਾਂਗੇ, ਜਿਹੜਾ ਕ੍ਰਮਵਾਰ ਇਕ ਪ੍ਰਣਾਲੀ ਤਿਆਰ ਕਰੇਗਾ ਜੋ ਮੰਜ਼ਿਲਾਂ ਸੁਝਾਅ ਦਿੰਦਾ ਹੈ, ਉਪਲਬਧਤਾ ਜਾਂਚਦਾ ਹੈ, ਅਤੇ ਲੋਜਿਸਟਿਕਸ ਸੰਭਾਲਦਾ ਹੈ।


## ਸੈਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਪੈਟਰਨ 1: ਸਾਫ਼ ਏਜੰਟ ਹਦਾਇਤਾਂ

ਸਭ ਤੋਂ ਪ੍ਰਭਾਵਸ਼ালী ਪੈਟਰਨ ਸਭ ਤੋਂ ਸਧਾਰਣ ਵੀ ਹੈ: ਆਪਣੇ ਏਜੰਟ ਲਈ ਸਾਫ਼, ਵਿਸਥਾਰਪੂਰਕ ਹਦਾਇਤਾਂ ਲਿਖਣਾ।

ਚੰਗੀਆਂ ਹਦਾਇਤਾਂ ਇਹਨਾਂ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੀਆਂ ਹਨ:
- **ਕੌਣ** ਏਜੰਟ ਹੈ (ਪੇਰੋਨਾ ਅਤੇ ਟੋਨ)
- **ਕੀ** ਇਹ ਕਰਨਾ ਚਾਹੀਦਾ ਹੈ (ਕਦਮ-ਬ ਕਦਮ ਜ਼ਿੰਮੇਵਾਰੀਆਂ)
- **ਕਿਵੇਂ** ਇਸ ਨੂੰ ਵਰਤਣਾ ਚਾਹੀਦਾ ਹੈ (ਬੰਨ੍ਹਣ ਜਾਂ ਸਟਾਈਲ)

ਹੇਠਾਂ, ਅਸੀਂ ਇੱਕ ਯਾਤਰਾ ਕੰਸੀਅਰਜ ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜਿਸਦੀ ਹਰ ਜਵਾਬ ਵਿੱਚ ਖਾਸ ਹਦਾਇਤਾਂ ਹੁੰਦੀਆਂ ਹਨ ਜੋ ਇਸਦੇ ਜਵਾਬਾਂ ਨੂੰ ਰੂਪ ਦਿੰਦੀਆਂ ਹਨ।


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

The key to enforcement is passing `response_format` when we run the agent. This forces the
model to return a validated `TravelRecommendations` object (available on `response.value`)
instead of free-form text. The `get_destination_details` tool also returns a typed
`DestinationRecommendation`, so the data stays structured end to end.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## ਪੈਟਰਨ 3: ਇਕੱਲਾ ਜ਼ਿੰਮੇਵਾਰੀ ਵਾਲੇ ਏਜੰਟ

ਜਟਿਲ ਕੰਮਾਂ ਨੂੰ ਬਹੁਤ ਸਾਰੇ ਕੇਂਦਰਿਤ ਏਜੰਟਾਂ ਵਿੱਚ ਵੰਡਣ ਨਾਲ ਫਾਇਦਾ ਹੁੰਦਾ ਹੈ, ਹਰ ਇੱਕ ਦੀ ਇੱਕ ਹੀ ਜ਼ਿੰਮੇਵਾਰੀ ਹੁੰਦੀ ਹੈ:

- ਇੱਕ **ਮੰਜ਼ਿਲ ਵਿਸ਼ੇਸ਼ਜ੍ਞ** ਜੋ ਸਥਾਨਾਂ ਅਤੇ ਉਪਲਬਧਤਾ ਬਾਰੇ ਜਾਣਦਾ ਹੈ
- ਇੱਕ **ਲੋਜਿਸਟਿਕਸ ਯੋਜਕ** ਜੋ ਉਡਾਣਾਂ, ਹੋਟਲਾਂ ਅਤੇ ਯਾਤਰਾ ਦੇ ਕਾਰਜ ਸੰਭਾਲਦਾ ਹੈ

ਇਹ ਸਾਫਟਵੇਅਰ ਇੰਜੀਨੀਅਰਿੰਗ ਦੇ ਸਿਧਾਂਤ *ਫਰਕ ਬੰਨਾ* ਦੀ ਨਕਲ ਕਰਦਾ ਹੈ — ਹਰ ਏਜੰਟ ਨੂੰ ਅਜ਼ਾਦਾਨਾ ਟੈਸਟ, ਰੱਖ-ਰਖਾਅ ਅਤੇ ਸੁਧਾਰ ਕਰਨਾ ਆਸਾਨ ਹੁੰਦਾ ਹੈ।


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਅਸੀਂ ਚਾਰਿਆ ਪਰਸਾਰਕ ਡਿਜ਼ਾਇਨ ਪੈਟਰਨ ਇੱਕ ਯਾਤਰਾ ਸਿਫਾਰਸ਼ੀ ਸੰਦਰਭ ਵਿੱਚ ਲਾਗੂ ਕੀਤੇ:

| ਪੈਟਰਨ | ਮੁੱਖ ਵਿਚਾਰ | ਲਾਭ |
|---|---|---|
| **ਸਪਸ਼ਟ ਹੁਕਮ** | ਪਹਿਚਾਣ, ਜਿੰਮੇਵਾਰੀਆਂ ਅਤੇ ਪਾਬੰਦੀਆਂ ਪਹਿਲਾਂ ਤੋਂ ਨਿਰਧਾਰਤ ਕਰੋ | ਸਥਿਰ, ਬ੍ਰਾਂਡ ਅਨੁਕੂਲ ਏਜੰਟ ਵਿਹਾਰ |
| **ਸੰਰਚਿਤ ਨਤੀਜਾ** | ਜਵਾਬ ਦੇ ਫਾਰਮੈਟ ਵਜੋਂ ਪਾਇਡੈਂਟਿਕ ਮਾਡਲਾਂ ਦੀ ਵਰਤੋਂ ਕਰੋ | ਪ੍ਰਮਾਣਿਤ, ਮਸ਼ੀਨ-ਪੜ੍ਹਨਯੋਗ ਨਤੀਜੇ |
| **ਇੱਕੱਲੀ ਜਿੰਮੇਵਾਰੀ** | ਹਰ ਏਜੰਟ ਨੂੰ ਇੱਕ ਕੇਂਦਰਿਤ ਕੰਮ ਦਿਓ | ਆਸਾਨ ਟੈਸਟਿੰਗ, ਰੱਖ-ਰਖਾਅ ਅਤੇ ਸੰਯੋਜਨ |

ਇਹ ਪੈਟਰਨ ਪ੍ਰਾਕ੍ਰਿਤਿਕ ਤੌਰ 'ਤੇ ਮਿਲ ਕੇ ਕੰਮ ਕਰਦੇ ਹਨ — ਤੁਸੀਂ ਸਪਸ਼ਟ ਹੁਕਮਾਂ ਨੂੰ ਸੰਰਚਿਤ ਨਤੀਜੇ ਨਾਲ ਇੱਕੱਲੀ ਜਿੰਮੇਵਾਰੀ ਵਾਲੇ ਏਜੰਟ ਅੰਦਰ ਮਿਲਾ ਸਕਦੇ ਹੋ, ਜਿਸ ਨਾਲ ਮਜ਼ਬੂਤ, ਉਤਪਾਦਨ-ਤਿਆਰ ਸਿਸਟਮ ਬਣਦੇ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
